# Preparar dados para exploração e modelagem

Este notebook baixa os arquivos mensais do VRA e gera as bases locais usadas pelos notebooks `01_entendimento_do_dataset.ipynb` e `02_modelagem_faixas_atraso.ipynb`.

Para datasets oficiais, imutáveis e vinculados a runs, use o materializador em `datasets/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent
if not (ROOT / 'scripts').exists():
    raise FileNotFoundError('Não encontrei a raiz do projeto com a pasta scripts/.')

INICIO_MES = '2024-01'
FIM_MES = '2025-12'
print('Raiz:', ROOT)
print('Período:', INICIO_MES, 'até', FIM_MES)

## Limpar arquivos brutos anteriores

Antes de baixar, removemos somente os CSVs VRA e arquivos temporários gerados anteriormente em `data/raw/`. Arquivos com outros nomes não são alterados.

In [ ]:
RAW_DIR = ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
arquivos_anteriores = sorted(list(RAW_DIR.glob('VRA_*.csv')) + list(RAW_DIR.glob('VRA_*.csv.download')))
for arquivo in arquivos_anteriores:
    arquivo.unlink()
    print('Removido:', arquivo)
print(f'{len(arquivos_anteriores)} arquivo(s) anterior(es) removido(s).')

## Baixar os arquivos mensais

O intervalo é inclusivo. O downloader informa o início e o sucesso de cada mês no output da célula.

In [ ]:
def executar(*argumentos):
    comando = [sys.executable, *argumentos]
    print('+', ' '.join(comando))
    subprocess.run(comando, cwd=ROOT, check=True)

executar(
    'scripts/baixar_amostra.py',
    '--inicio-mes', INICIO_MES,
    '--fim-mes', FIM_MES,
    '--output-dir', 'data/raw',
    '--metadata-output', 'data/amostra_metadata.json',
    '--sobrescrever',
)

## Auditar os arquivos brutos

A auditoria verifica a estrutura dos CSVs baixados, incluindo colunas, duplicatas e valores ausentes. Ela gera `data/auditoria_amostra.json`.

In [ ]:
executar(
    'scripts/auditar_amostra_v2.py',
    '--input-dir', 'data/raw',
    '--output', 'data/auditoria_amostra.json',
)

## Gerar o dataset derivado

Esta etapa normaliza os nomes, datas e tipos dos arquivos brutos e gera `data/voos_vra_derivados.csv`, que é usado pelo notebook de entendimento. O script:

- lê todos os arquivos `VRA_*.csv` encontrados em `data/raw/`;
- valida e seleciona as colunas esperadas do VRA e renomeia os campos para nomes padronizados;
- converte as quatro colunas de data e hora para tipos de data do pandas;
- calcula os atrasos de partida e chegada em minutos;
- cria os indicadores booleanos de voo realizado, cancelado e atraso de pelo menos 15 minutos;
- deixa os indicadores de atraso ausentes para voos que não foram realizados;
- registra o arquivo e a linha de origem para rastreabilidade;
- concatena os meses em uma única tabela e salva o CSV derivado.

O script não cria ainda a coluna `faixa_atraso`; essa materialização ocorre na etapa seguinte, ao gerar a base de modelagem.

In [ ]:
executar(
    'scripts/preparar_dados_v2.py',
    '--input-dir', 'data/raw',
    '--output', 'data/voos_vra_derivados.csv',
)

## Materializar a base de modelagem e as faixas de atraso

Esta etapa lê o dataset derivado, mantém somente os voos realizados com atraso de chegada calculável, cria a coluna-alvo `faixa_atraso`, gera as variáveis temporais e salva `data/modelagem_faixas_atraso.csv`, que será usado pelos modelos. O output informa as linhas não elegíveis, os critérios aplicados, a redução líquida de colunas, as colunas brutas excluídas e as colunas criadas. Também gera a distribuição mensal e exibe a quantidade e o percentual de registros em cada uma das seis classes.

In [ ]:
executar(
    'scripts/preparar_modelagem.py',
    '--input', 'data/voos_vra_derivados.csv',
    '--output', 'data/modelagem_faixas_atraso.csv',
    '--monthly-output', 'data/distribuicao_mensal_faixas_atraso.csv',
)

## Preparar a conferência

Primeiro importamos o pandas e registramos os nomes das seis classes para que a tabela final seja apresentada em ordem e com descrições legíveis.

In [ ]:
import pandas as pd

FAIXAS = {
    0: 'Pontual ou antecipado',
    1: 'Atraso inferior a 15 min',
    2: 'Atraso de 15 a 30 min',
    3: 'Atraso superior a 30 até 45 min',
    4: 'Atraso superior a 45 até 60 min',
    5: 'Atraso superior a 60 min',
}

## Carregar e conferir as bases

Agora carregamos o dataset derivado e a base de modelagem e mostramos suas dimensões. Isso confirma que os arquivos esperados foram gerados antes de analisar as classes.

In [ ]:
derivado = pd.read_csv(ROOT / 'data' / 'voos_vra_derivados.csv', low_memory=False)
modelagem = pd.read_csv(ROOT / 'data' / 'modelagem_faixas_atraso.csv', low_memory=False)
print('data/voos_vra_derivados.csv:', derivado.shape)
print('data/modelagem_faixas_atraso.csv:', modelagem.shape)

## Distribuição das faixas de atraso

Por fim, contamos os registros de cada `faixa_atraso` e calculamos seu percentual na base de modelagem. Essa tabela permite verificar o desbalanceamento entre as classes antes do treinamento.

In [ ]:
distribuicao = (
    modelagem['faixa_atraso'].value_counts()
    .reindex(FAIXAS.keys(), fill_value=0)
    .rename('quantidade')
    .to_frame()
)
distribuicao.index.name = 'codigo'
distribuicao['classificacao'] = distribuicao.index.map(FAIXAS)
distribuicao['percentual'] = (100 * distribuicao['quantidade'] / len(modelagem)).round(2)
distribuicao[['classificacao', 'quantidade', 'percentual']]